# Train Weed Risk and Irrigation Optimizer Models

This notebook trains two **model-driven** modules for satellite recommendations:
1. `weed_risk_model.joblib` (classifier with calibrated probability)
2. `irrigation_response_model.joblib` (regressor used for optimization over candidate actions)

It also explains how to collect and structure data for production training.

## 1) Required Data Files

Place two CSV files in `backend/ml_service/data/training/`:

### A. `weed_dataset.csv`
Recommended columns:
- `ndvi`, `evi`, `savi`, `ndwi`, `ndre`, `gndvi`
- `month`, `growth_stage_code`
- `rain_next_48h`, `avg_humidity`, `max_wind_speed`, `hot_days`
- Optional metadata: `district`, `field_id`, `analysis_date`
- Label: `weed_high` (0/1)

### B. `irrigation_dataset.csv`
Recommended columns:
- Same state features as above
- Action features: `irrigation_mm_applied`, `irrigation_method`
- Outcome: either `stress_delta_7d` directly, or (`stress_before`, `stress_after_7d`)

If labels are missing, the notebook builds temporary weak labels so you can iterate today.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    mean_absolute_error, r2_score
)

import joblib

In [ ]:
ROOT = Path('..').resolve()  # expected: backend/ml_service
DATA_DIR = ROOT / 'data' / 'training'
MODEL_DIR = ROOT.parent / 'ml_models' / 'satellite_models'

WEED_DATA_PATH = DATA_DIR / 'weed_dataset.csv'
IRRIG_DATA_PATH = DATA_DIR / 'irrigation_dataset.csv'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR:', DATA_DIR)
print('MODEL_DIR:', MODEL_DIR)

In [ ]:
# Create templates if files are missing
if not WEED_DATA_PATH.exists():
    pd.DataFrame(columns=[
        'ndvi','evi','savi','ndwi','ndre','gndvi',
        'month','growth_stage_code','rain_next_48h','avg_humidity','max_wind_speed','hot_days',
        'district','field_id','analysis_date','weed_high'
    ]).to_csv(WEED_DATA_PATH, index=False)

if not IRRIG_DATA_PATH.exists():
    pd.DataFrame(columns=[
        'ndvi','evi','savi','ndwi','ndre','gndvi',
        'month','growth_stage_code','rain_next_48h','avg_humidity','max_wind_speed','hot_days',
        'soil_clay','soil_ec','soil_ph',
        'irrigation_mm_applied','irrigation_method','stress_before','stress_after_7d','stress_delta_7d'
    ]).to_csv(IRRIG_DATA_PATH, index=False)

print('Templates are ready.')

In [ ]:
weed_df = pd.read_csv(WEED_DATA_PATH)
irrig_df = pd.read_csv(IRRIG_DATA_PATH)

print('weed_df shape:', weed_df.shape)
print('irrig_df shape:', irrig_df.shape)
display(weed_df.head(3))
display(irrig_df.head(3))

## 2) Weed Model Training

If `weed_high` is missing, we bootstrap weak labels for quick MVP training.
Replace these with field-scouted labels for production reliability.

In [ ]:
required_weed_features = [
    'ndvi','evi','savi','ndwi','ndre','gndvi',
    'month','growth_stage_code','rain_next_48h','avg_humidity','max_wind_speed','hot_days'
]

for c in required_weed_features:
    if c not in weed_df.columns:
        weed_df[c] = np.nan

if 'weed_high' not in weed_df.columns or weed_df['weed_high'].isna().all():
    # Weak label for bootstrap only
    weed_signal = (
        (weed_df['ndvi'].fillna(0.3) < 0.35).astype(int)
        + (weed_df['evi'].fillna(0.25) < 0.30).astype(int)
        + (weed_df['ndwi'].fillna(-0.1) > -0.18).astype(int)
        + (weed_df['growth_stage_code'].fillna(3).isin([1,2,3])).astype(int)
    )
    weed_df['weed_high'] = (weed_signal >= 3).astype(int)

weed_df['weed_high'] = weed_df['weed_high'].astype(int)
weed_df = weed_df.dropna(subset=['weed_high']).copy()

print('weed label distribution:\n', weed_df['weed_high'].value_counts(dropna=False))

In [ ]:
weed_num = [
    'ndvi','evi','savi','ndwi','ndre','gndvi',
    'month','growth_stage_code','rain_next_48h','avg_humidity','max_wind_speed','hot_days'
]
weed_cat = [c for c in ['district'] if c in weed_df.columns]

X = weed_df[weed_num + weed_cat].copy()
y = weed_df['weed_high'].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

pre = ColumnTransformer([
    ('num', num_pipe, weed_num),
    ('cat', cat_pipe, weed_cat),
], remainder='drop')

base_clf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

cal_clf = CalibratedClassifierCV(base_clf, method='isotonic', cv=3)
weed_model = Pipeline([('pre', pre), ('model', cal_clf)])
weed_model.fit(X_train, y_train)

p_val = weed_model.predict_proba(X_val)[:, 1]
threshold_grid = np.arange(0.20, 0.86, 0.01)
f1_grid = [f1_score(y_val, (p_val >= t).astype(int)) for t in threshold_grid]
best_idx = int(np.argmax(f1_grid))
best_thr = float(threshold_grid[best_idx])
y_hat = (p_val >= best_thr).astype(int)

weed_metrics = {
    'auc': float(roc_auc_score(y_val, p_val)),
    'f1': float(f1_score(y_val, y_hat)),
    'precision': float(precision_score(y_val, y_hat, zero_division=0)),
    'recall': float(recall_score(y_val, y_hat, zero_division=0)),
    'best_threshold': best_thr,
    'train_rows': int(len(X_train)),
    'val_rows': int(len(X_val)),
}

print('Weed model metrics:', json.dumps(weed_metrics, indent=2))

In [ ]:
weed_model_path = MODEL_DIR / 'weed_risk_model.joblib'
weed_metrics_path = MODEL_DIR / 'weed_risk_metrics.json'

joblib.dump({'pipeline': weed_model, 'threshold': weed_metrics['best_threshold']}, weed_model_path)
weed_metrics_path.write_text(json.dumps(weed_metrics, indent=2), encoding='utf-8')

print('Saved:', weed_model_path)
print('Saved:', weed_metrics_path)

## 3) Irrigation Response Model Training

This model predicts expected stress improvement for a given irrigation action,
then your backend optimizer can search over candidate actions and pick the best one.

In [ ]:
required_irrig_cols = [
    'ndvi','evi','savi','ndwi','ndre','gndvi',
    'month','growth_stage_code','rain_next_48h','avg_humidity','max_wind_speed','hot_days',
    'soil_clay','soil_ec','soil_ph','irrigation_mm_applied','irrigation_method'
]

for c in required_irrig_cols:
    if c not in irrig_df.columns:
        irrig_df[c] = np.nan

if 'stress_delta_7d' not in irrig_df.columns or irrig_df['stress_delta_7d'].isna().all():
    if {'stress_before', 'stress_after_7d'}.issubset(irrig_df.columns):
        irrig_df['stress_delta_7d'] = irrig_df['stress_before'] - irrig_df['stress_after_7d']
    else:
        # Weak fallback target (MVP only)
        irrig_df['stress_delta_7d'] = (
            0.015 * irrig_df['irrigation_mm_applied'].fillna(0)
            + 0.20 * (irrig_df['ndwi'].fillna(-0.1) < -0.1).astype(float)
            - 0.08 * (irrig_df['rain_next_48h'].fillna(0) > 10).astype(float)
        )

irrig_df = irrig_df.dropna(subset=['stress_delta_7d']).copy()
print('Irrigation rows for training:', len(irrig_df))

In [ ]:
irrig_num = [
    'ndvi','evi','savi','ndwi','ndre','gndvi',
    'month','growth_stage_code','rain_next_48h','avg_humidity','max_wind_speed','hot_days',
    'soil_clay','soil_ec','soil_ph','irrigation_mm_applied'
]
irrig_cat = [c for c in ['irrigation_method', 'district'] if c in irrig_df.columns]

Xr = irrig_df[irrig_num + irrig_cat].copy()
yr = irrig_df['stress_delta_7d'].copy()

Xr_train, Xr_val, yr_train, yr_val = train_test_split(
    Xr, yr, test_size=0.2, random_state=42
)

num_pipe_r = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe_r = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

pre_r = ColumnTransformer([
    ('num', num_pipe_r, irrig_num),
    ('cat', cat_pipe_r, irrig_cat),
], remainder='drop')

irrig_reg = RandomForestRegressor(
    n_estimators=600,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

irrig_model = Pipeline([('pre', pre_r), ('model', irrig_reg)])
irrig_model.fit(Xr_train, yr_train)

yr_pred = irrig_model.predict(Xr_val)
irrig_metrics = {
    'mae': float(mean_absolute_error(yr_val, yr_pred)),
    'r2': float(r2_score(yr_val, yr_pred)),
    'train_rows': int(len(Xr_train)),
    'val_rows': int(len(Xr_val)),
}

print('Irrigation model metrics:', json.dumps(irrig_metrics, indent=2))

In [ ]:
irrig_model_path = MODEL_DIR / 'irrigation_response_model.joblib'
irrig_metrics_path = MODEL_DIR / 'irrigation_response_metrics.json'

joblib.dump({'pipeline': irrig_model, 'candidate_mm': [0, 10, 20, 30, 40]}, irrig_model_path)
irrig_metrics_path.write_text(json.dumps(irrig_metrics, indent=2), encoding='utf-8')

print('Saved:', irrig_model_path)
print('Saved:', irrig_metrics_path)

## 4) How Optimizer Uses the Irrigation Model

At inference, for a given field state, we simulate candidate irrigation actions and choose the one with max score:

`score = predicted_stress_delta - lambda_water * irrigation_mm_applied`

This keeps recommendations data-driven and cost-aware.

In [ ]:
def optimize_irrigation_action(context_row: dict, model_pack: dict, lambda_water: float = 0.003):
    pipe = model_pack['pipeline']
    candidate_mm = model_pack.get('candidate_mm', [0, 10, 20, 30, 40])
    methods = context_row.get('method_candidates', ['furrow', 'flood', 'sprinkler'])

    rows = []
    for mm in candidate_mm:
        for m in methods:
            r = dict(context_row)
            r['irrigation_mm_applied'] = mm
            r['irrigation_method'] = m
            rows.append(r)

    cand_df = pd.DataFrame(rows)
    pred_delta = pipe.predict(cand_df)
    score = pred_delta - lambda_water * cand_df['irrigation_mm_applied'].values

    best_i = int(np.argmax(score))
    best = cand_df.iloc[best_i].to_dict()
    best['predicted_stress_delta'] = float(pred_delta[best_i])
    best['optimizer_score'] = float(score[best_i])
    return best

# Example usage
loaded_irrig = joblib.load(irrig_model_path)
example_context = {
    'ndvi': 0.28, 'evi': 0.18, 'savi': 0.21, 'ndwi': -0.14, 'ndre': 0.15, 'gndvi': 0.24,
    'month': 2, 'growth_stage_code': 3,
    'rain_next_48h': 1.2, 'avg_humidity': 39.0, 'max_wind_speed': 3.8, 'hot_days': 2,
    'soil_clay': 17.0, 'soil_ec': 1.1, 'soil_ph': 7.7,
    'method_candidates': ['furrow', 'flood', 'sprinkler']
}

best_action = optimize_irrigation_action(example_context, loaded_irrig)
best_action

## 5) How to Get Data (Practical Plan)

### Weed model data
1. Export satellite analysis logs (indices + weather + stage + date + field_id).
2. Add field scouting form for each visit:
   - weed_cover_pct
   - dominant weed type
   - action taken (yes/no)
3. Create label: `weed_high = 1` if weed_cover_pct >= threshold (for example 10%).
4. Join scouting records to nearest satellite timestamp (same field_id, +/-3 days).

### Irrigation model data
1. For each irrigation event capture:
   - irrigation_mm_applied
   - irrigation_method
   - date/time
2. Capture stress outcome:
   - stress_before
   - stress_after_7d (or next valid satellite pass)
3. Compute target:
   - `stress_delta_7d = stress_before - stress_after_7d`
4. Merge weather + soil + growth-stage context from your API logs.

### Minimum dataset size (MVP)
- Weed: 500+ rows (100+ true positives preferred)
- Irrigation: 800+ action-outcome rows

### Quality rules
- Keep strict train/validation split by date or field to avoid leakage.
- Re-train monthly as new field labels come in.
- Store metrics JSON and model version with each training run.